# 02 · TMDB exploration: what will matching Netflix titles to TMDB involve?

**Goal:** before writing `src/tmdb.py`, learn what the matching problem looks like on real titles:

| Section | Question | Feeds into |
| --- | --- | --- |
| 1 | How many titles and TMDB lookups are there? | How long a full run takes, caching |
| 2 | What do the season titles look like? | Rules for turning `season_title` into a season number |
| 3–4 | What does TMDB return for a sample of ~20 titles? | Seeing the tricky cases first-hand |
| 5 | Does a simple matching rule pick the right result? | The matching logic in `src/tmdb.py` |
| 6 | Do TMDB season runtimes agree with Netflix's? | Evidence that the matches are right |

**Before running:**
- `.env` at the repo root contains `TMDB_API_KEY=...`
- Add this line to `.gitignore`, so the TMDB responses cached by this notebook are never committed:
  ```
  /data/external/
  ```

Every TMDB response is saved under `data/external/tmdb_cache/exploration/`, so rerunning the notebook doesn't call TMDB again.

## 0 · Setup

In [60]:
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

# Let the notebook import the project's own code (src/), whether it runs from notebooks/ or the repo root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
from src.features import add_title_key
from src.ingest import latest_snapshot, load_raw

load_dotenv(REPO_ROOT / ".env")
TMDB_KEY = os.getenv("TMDB_API_KEY")
assert TMDB_KEY, "TMDB_API_KEY not found: check .env is at the repo root and contains TMDB_API_KEY=..."

CACHE_DIR = REPO_ROOT / "data" / "external" / "tmdb_cache" / "exploration"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

In [61]:
# TMDB accepts two kinds of key: the short "API Key" (sent as a parameter) or the long
# "API Read Access Token" starting with eyJ (sent as a header). Either works here.
session = requests.Session()
if TMDB_KEY.startswith("eyJ"):
    session.headers["Authorization"] = f"Bearer {TMDB_KEY}"
    AUTH = {}
else:
    AUTH = {"api_key": TMDB_KEY}

API_CALLS = {"network": 0, "cache": 0}


def tmdb_get(path, **params):
    # Call a TMDB endpoint, e.g. tmdb_get("/search/tv", query="Wednesday").
    # Each response is saved as a JSON file; the same call later reads the file instead of calling TMDB.
    request_id = path + "?" + "&".join(f"{k}={v}" for k, v in sorted(params.items()))
    readable = re.sub(r"[^A-Za-z0-9]+", "_", request_id)[:80]
    cache_file = CACHE_DIR / f"{readable}_{hashlib.md5(request_id.encode()).hexdigest()[:10]}.json"
    if cache_file.exists():
        API_CALLS["cache"] += 1
        return json.loads(cache_file.read_text(encoding="utf-8"))

    response = session.get(f"https://api.themoviedb.org/3{path}", params={**params, **AUTH}, timeout=30)
    API_CALLS["network"] += 1
    time.sleep(0.05)  # stay well under TMDB's rate limit
    if response.status_code == 404:
        return None  # e.g. a season number the show doesn't have
    response.raise_for_status()
    cache_file.write_text(response.text, encoding="utf-8")
    return response.json()


print("Connection check:", tmdb_get("/configuration") is not None)

Connection check: True


## 1 · What needs looking up?

One row per **title** (list + show + season), with the week it first charted. That date matters: Netflix's file has no release year, so "released before it first charted" is the main clue for picking the right TMDB result.

In [62]:
raw = add_title_key(load_raw(latest_snapshot()))
raw["is_tv"] = raw["category"].str.startswith("TV")

titles = (
    raw.groupby("title_key")
    .agg(
        category=("category", "first"),
        show_title=("show_title", "first"),
        season_title=("season_title", "first"),
        is_tv=("is_tv", "first"),
        first_week=("week", "min"),
        weeks_charted=("week", "nunique"),
        netflix_runtime=("runtime", lambda s: s[s > 0].max()),  # hours; the full season once all episodes are out
    )
    .reset_index()
)

unique_films = titles.loc[~titles["is_tv"], "show_title"].nunique()
unique_shows = titles.loc[titles["is_tv"], "show_title"].nunique()
tv_seasons = int(titles["is_tv"].sum())

print(f"Titles (list + show + season): {len(titles):,}")
print(f"Unique films: {unique_films:,}   Unique TV shows: {unique_shows:,}   TV seasons: {tv_seasons:,}")
print(
    f"\nA full run needs roughly {unique_films:,} film searches + {unique_films:,} film detail calls"
    f" + {unique_shows:,} TV searches + {tv_seasons:,} season calls"
    f" = ~{2 * unique_films + unique_shows + tv_seasons:,} requests (a few minutes, once; then cached)."
)
titles.groupby("category").agg(titles=("title_key", "size"), unique_shows=("show_title", "nunique"))

Titles (list + show + season): 4,032
Unique films: 2,242   Unique TV shows: 1,296   TV seasons: 1,784

A full run needs roughly 2,242 film searches + 2,242 film detail calls + 1,296 TV searches + 1,784 season calls = ~7,564 requests (a few minutes, once; then cached).


,titles,unique_shows
category,,
Films (English),1101,1101
Films (Non-English),1147,1147
TV (English),958,622
TV (Non-English),826,674


## 2 · Turning `season_title` into a season number

TMDB stores TV shows by season number, so each `season_title` has to become a number. This draft rule handles the common patterns; the table shows how many titles each pattern covers, and the **other** list shows titles the rule doesn't understand yet. Decide what to do with those.

In [63]:
def parse_season(show_title, season_title):
    # Return (season_number, pattern_name). season_number is None when the title alone can't tell.
    if pd.isna(season_title) or str(season_title).strip() in ("", "N/A"):
        return None, "no season title"
    s = str(season_title)
    rest = s[len(show_title):].strip(" :-") if s.startswith(show_title) else s  # drop the show name

    patterns = [
        (r"\bSeason (\d+)", "Season N"),
        (r"\bPart (\d+)", "Part N"),
        (r"\b(?:Volume|Vol\.) ?(\d+)", "Volume N"),
        (r"\bBook (\d+)", "Book N"),
        (r"\bChapter (\d+)", "Chapter N"),
    ]
    for regex, name in patterns:
        match = re.search(regex, s, flags=re.IGNORECASE)
        if match:
            if name == "Part N":
                return None, name  # parts are usually halves of one TMDB season: look it up instead
            return int(match.group(1)), name
    if re.search(rf"\b(?:{MONTHS}) \d{{1,2}}, \d{{4}}", s):
        return None, "Dated episode"
    if re.search(r"Limited Series|Miniseries|Mini-Series", s, flags=re.IGNORECASE):
        return 1, "Limited series"
    if rest == "":
        return 1, "Same as show title"
    if re.fullmatch(r"\d{1,2}", rest):
        return int(rest), "Show name + number"  # e.g. "Stranger Things 4"
    return None, "other"


tv = titles[titles["is_tv"]].copy()
tv[["season_number", "season_pattern"]] = [
    parse_season(show, season) for show, season in zip(tv["show_title"], tv["season_title"])
]

summary = tv.groupby("season_pattern").agg(
    titles=("title_key", "size"),
    example=("season_title", "first"),
).sort_values("titles", ascending=False)
summary["share"] = (summary["titles"] / len(tv)).map("{:.1%}".format)
display(summary)

,titles,example,share
season_pattern,,,
Season N,1199,1883: Season 1,67.2%
Limited series,335,7 Bears: Limited Series,18.8%
no season title,86,NaN,4.8%
Dated episode,85,"Raw: April 13, 2026",4.8%
other,44,Ancient Apocalypse: The Americas,2.5%
Part N,19,Disenchantment: Part 4,1.1%
Volume N,6,Little Angel: Volume 1,0.3%
Chapter N,5,The Creature Cases: Chapter 1,0.3%
Show name + number,5,Stranger Things 2,0.3%


In [64]:
# Patterns worth a closer look: "Part N" (is Part 2 its own TMDB season, or the second half of one?)
# and "other" (titles the rule can't parse yet).
for pattern in ["Part N", "other"]:
    rows = tv[tv["season_pattern"] == pattern]
    print(f"\n{pattern}: {len(rows)} titles")
    display(rows[["show_title", "season_title", "first_week"]].head(30))


Part N: 19 titles


,show_title,season_title,first_week
2441,Disenchantment,Disenchantment: Part 4,2022-02-13
2442,Disenchantment,Disenchantment: Part 5,2023-09-03
2564,Inside Job,Inside Job: Part 1,2021-10-31
2985,That '90s Show,That '90s Show: Part 1,2023-01-22
2986,That '90s Show,That '90s Show: Part 2,2024-07-07
3090,The Upshaws,The Upshaws: Part 4,2023-08-20
3230,Alchemy of Souls,Alchemy of Souls: Part 1,2022-07-03
3231,Alchemy of Souls,Alchemy of Souls: Part 2,2022-12-18
3256,Attack on Titan,Attack on Titan: The Final Season Part 3,2023-11-12
3385,Dog Days Out,Dog Days Out: Part 1,2025-02-16



other: 44 titles


,show_title,season_title,first_week
2294,Ancient Apocalypse,Ancient Apocalypse: The Americas,2024-10-20
2332,Bebefinn Playtime,Bebefinn Playtime: 시즌 1,2023-12-31
2417,DAHMER,DAHMER: Monster: The Jeffrey Dahmer Story,2022-09-25
2494,Funny AF with Kevin Hart,Funny AF with Kevin Hart: The Live Semifinals,2026-05-10
2540,Homicide,Homicide: Los Angeles,2024-07-21
2577,Is It Cake?,"Is It Cake?: Is It Cake, Too?",2023-07-09
2635,Love Is Blind,Love Is Blind: S10: Ohio,2026-02-15
2636,Love Is Blind,Love Is Blind: S11: Boston,2026-10-18
2637,Love Is Blind,Love Is Blind: S1: Atlanta,2021-08-01
2638,Love Is Blind,Love Is Blind: S2: Chicago,2022-02-13


In [65]:
# Worth examining the repeated appearance of Raw, likely the weekly episodes of WWE Raw. 
# Netflix lists each weekly episode as its own title, named by its air date.
# This matters for our models because Each episode charts for about one week, then the next episode replaces it. 
# So every Raw row gets labeled “didn’t survive”, even though the show itself is on the chart every week.

MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
is_dated = titles["season_title"].astype(str).str.contains(rf"\b(?:{MONTHS}) \d{{1,2}}, \d{{4}}")
dated = titles[is_dated]
print(f"{len(dated)} dated-episode titles ({len(dated) / len(titles):.1%} of all titles)")
display(dated.groupby(["category", "show_title"]).agg(
    episodes=("title_key", "size"),
    first=("first_week", "min"),
    last=("first_week", "max"),
    median_weeks_charted=("weeks_charted", "median"),
))

85 dated-episode titles (2.1% of all titles)


,,episodes,first,last,median_weeks_charted
category,show_title,,,,
TV (English),Raw,85,2025-01-12,2026-10-18,1.0


## 3 · Pick ~20 titles to look up by hand

Five random titles from each list. Add any titles you're curious about to `EXTRA_TITLE_KEYS`, using the `title_key` format shown in section 1, for example `"TV (Non-English) | Squid Game | Squid Game: Season 1"`.

In [66]:
PER_LIST = 5
# 10 TV titles that have a Netflix runtime, so section 6 can cross-check TMDB against them
EXTRA_TITLE_KEYS = (
    titles.loc[titles["is_tv"] & titles["netflix_runtime"].notna(), "title_key"]
    .sample(10, random_state=3)
    .tolist()
)

sample = titles.groupby("category", group_keys=False).sample(PER_LIST, random_state=7)
extra = titles[titles["title_key"].isin(EXTRA_TITLE_KEYS)]
sample = pd.concat([sample, extra]).drop_duplicates("title_key").reset_index(drop=True)
sample[["category", "show_title", "season_title", "first_week", "weeks_charted"]]

,category,show_title,season_title,first_week,weeks_charted
0,Films (English),Spider-Man: No Way Home,NaN,2022-07-17,2
1,Films (English),Monsters vs. Aliens,NaN,2026-07-26,1
2,Films (English),Sweet Bobby: My Catfish Nightmare,NaN,2024-10-20,2
3,Films (English),Untold: The Liver King,NaN,2025-05-18,1
4,Films (English),The Boss Baby: Family Business,NaN,2023-12-31,1
5,Films (Non-English),Tadeo Jones 2: El secreto del Rey Midas,NaN,2022-07-03,2
6,Films (Non-English),Blasted,NaN,2022-07-03,2
7,Films (Non-English),Baramulla,NaN,2025-11-09,2
8,Films (Non-English),Under the Amalfi Sun,NaN,2022-07-17,2
9,Films (Non-English),Youth,NaN,2026-04-19,2


## 4 · What does TMDB return?

For each sampled title: the top search results, with their release (or first-air) date and how many days before the title first charted on Netflix. A negative number means TMDB says it was released *after* it charted, which usually means a wrong result.

`popularity` is TMDB's current popularity. It's fine for choosing between search results, but it must **never** become a model feature: it's today's value, even for a title from 2021.

In [67]:
def search_candidates(row, n=5):
    # Top n TMDB search results for one Netflix title, with dates relative to when it first charted.
    kind = "tv" if row["is_tv"] else "movie"
    result = tmdb_get(f"/search/{kind}", query=row["show_title"], include_adult="false", language="en-US") or {}
    rows = []
    for r in result.get("results", [])[:n]:
        date = pd.to_datetime(r.get("first_air_date") if row["is_tv"] else r.get("release_date"), errors="coerce")
        rows.append(
            {
                "tmdb_id": r["id"],
                "name": r.get("name") or r.get("title"),
                "original_name": r.get("original_name") or r.get("original_title"),
                "date": date,
                "days_before_charting": (row["first_week"] - date).days if pd.notna(date) else None,
                "language": r.get("original_language"),
                "popularity": r.get("popularity"),
            }
        )
    return pd.DataFrame(rows)


candidates = {}
for _, row in sample.iterrows():
    candidates[row["title_key"]] = search_candidates(row)
    print(f"\n{row['title_key']}   (first charted {row['first_week'].date()})")
    cands = candidates[row["title_key"]]
    display(cands if not cands.empty else "No results")

print(f"\nTMDB requests: {API_CALLS['network']} from the network, {API_CALLS['cache']} from the cache")


Films (English) | Spider-Man: No Way Home | N/A   (first charted 2022-07-17)


,tmdb_id,name,original_name,date,days_before_charting,language,popularity
0,634649,Spider-Man: No Way Home,Spider-Man: No Way Home,2021-12-15,214,en,39.3247
1,961651,Spider-Man: All Roads Lead to No Way Home,Spider-Man: All Roads Lead to No Way Home,2022-05-03,75,en,2.3388



Films (English) | Monsters vs. Aliens | N/A   (first charted 2026-07-26)


,tmdb_id,name,original_name,date,days_before_charting,language,popularity
0,15512,Monsters vs Aliens,Monsters vs Aliens,2009-03-19,6338,en,9.4252
1,294830,Monsters vs Aliens: Creature Features,Monsters vs Aliens: Creature Features,2014-10-01,4316,en,0.6286
2,486327,Monsters vs Aliens Stories,Monsters vs Aliens Stories,2013-01-01,4954,en,0.7942
3,376196,Monsters Vs Aliens: Cloning Around,Monsters Vs Aliens: Cloning Around,2013-10-15,4667,en,0.7335
4,1031810,Monsters vs. Aliens: Supersonic Joyride,Monsters vs. Aliens: Supersonic Joyride,2014-03-07,4524,en,0.1560



Films (English) | Sweet Bobby: My Catfish Nightmare | N/A   (first charted 2024-10-20)


,tmdb_id,name,original_name,date,days_before_charting,language,popularity
0,1361048,Sweet Bobby: My Catfish Nightmare,Sweet Bobby: My Catfish Nightmare,2024-10-15,5,en,1.2393


... (51 outputs left out)

,tmdb_id,name,original_name,date,days_before_charting,language,popularity
0,127865,The Chestnut Man,Kastanjemanden,2021-09-29,4,da,10.3055



TV (Non-English) | Valeria | Valeria: Season 3   (first charted 2023-06-04)


,tmdb_id,name,original_name,date,days_before_charting,language,popularity
0,101918,Valeria,Valeria,2020-05-08,1122,es,6.2608
1,222929,Valeria,Valeria,2008-03-10,5564,es,2.4481
2,122320,Valeria y Maximiliano,Valeria y Maximiliano,1991-12-23,11486,es,4.0908
3,229051,Valeria medico legale,Valeria medico legale,2000-04-25,8440,it,1.2367
4,307290,Valeria Mazza: Un sueño dorado,Valeria Mazza: Un sueño dorado,2023-06-16,-12,es,0.1189



TMDB requests: 0 from the network, 31 from the cache


## 5 · Does a simple matching rule pick the right one?

Draft rule, tried in order:
1. **Exact title match, released before it first charted.** For films, take the most recent one (catalog films can be old, new releases are recent). For TV, take the most popular, since the show may have started years before this season.
2. **Any result released before it first charted,** chosen the same way.
3. **Exact title match** with an unknown date.
4. Otherwise, **the top search result** (low confidence).

Check the table by eye: is each match the right title? Rows in the flagged list deserve a closer look.

In [69]:
def normalize(title):
    # Lowercase and drop punctuation and spaces, so "Spiderman" == "Spider-Man" and "&" == "and"
    if not isinstance(title, str):
        return ""
    return re.sub(r"[^\w]", "", title.casefold().replace("&", "and"))


def same_title(a, b):
    return normalize(a) != "" and normalize(a) == normalize(b)


def pick_match(row, cands):
    if cands.empty:
        return None, "no results"
    c = cands.copy()
    c["exact_title"] = [
        same_title(row["show_title"], n) or same_title(row["show_title"], o)
        for n, o in zip(c["name"], c["original_name"])
    ]
    c["released_before"] = c["days_before_charting"].fillna(-1) >= 0
    # Among exact matches: films -> most recent release (remakes share titles); TV -> most popular.
    exact_order = ("popularity", False) if row["is_tv"] else ("days_before_charting", True)
    most_popular = ("popularity", False)

    steps = [
        (c["exact_title"] & c["released_before"], "1 exact title, released before", exact_order),
        (c["released_before"], "2 released before", most_popular),
        (c["exact_title"], "3 exact title, date unknown/after", most_popular),
    ]
    for mask, rule, (column, ascending) in steps:
        if mask.any():
            return c[mask].sort_values(column, ascending=ascending).iloc[0], rule
    return c.iloc[0], "4 top search result"


matches = []
for _, row in sample.iterrows():
    best, rule = pick_match(row, candidates[row["title_key"]])
    matches.append(
        {
            "category": row["category"],
            "netflix_title": row["show_title"],
            "season_title": row["season_title"],
            "first_week": row["first_week"].date(),
            "tmdb_name": None if best is None else best["name"],
            "tmdb_date": None if best is None or pd.isna(best["date"]) else best["date"].date(),
            "days_before": None if best is None else best["days_before_charting"],
            "language": None if best is None else best["language"],
            "tmdb_id": None if best is None else best["tmdb_id"],
            "rule": rule,
            "results": len(candidates[row["title_key"]]),
        }
    )
matches = pd.DataFrame(matches)
display(matches)
print("\nHow often each rule decided:")
display(matches["rule"].value_counts())

,category,netflix_title,season_title,first_week,tmdb_name,tmdb_date,days_before,language,tmdb_id,rule,results
0,Films (English),Spider-Man: No Way Home,NaN,2022-07-17,Spider-Man: No Way Home,2021-12-15,214.0,en,634649.0,"1 exact title, released before",2
1,Films (English),Monsters vs. Aliens,NaN,2026-07-26,Monsters vs Aliens,2009-03-19,6338.0,en,15512.0,"1 exact title, released before",5
2,Films (English),Sweet Bobby: My Catfish Nightmare,NaN,2024-10-20,Sweet Bobby: My Catfish Nightmare,2024-10-15,5.0,en,1361048.0,"1 exact title, released before",1
3,Films (English),Untold: The Liver King,NaN,2025-05-18,Untold: The Liver King,2025-05-12,6.0,en,1465024.0,"1 exact title, released before",1
4,Films (English),The Boss Baby: Family Business,NaN,2023-12-31,The Boss Baby: Family Business,2021-07-01,913.0,en,459151.0,"1 exact title, released before",1
5,Films (Non-English),Tadeo Jones 2: El secreto del Rey Midas,NaN,2022-07-03,"Tad, the Lost Explorer, and the Secret of King Midas",2017-03-25,1926.0,es,452773.0,"1 exact title, released before",1
6,Films (Non-English),Blasted,NaN,2022-07-03,Blasted,2022-06-28,5.0,no,660267.0,"1 exact title, released before",5
7,Films (Non-English),Baramulla,NaN,2025-11-09,Baramulla,2025-11-06,3.0,hi,1561969.0,"1 exact title, released before",1
8,Films (Non-English),Under the Amalfi Sun,NaN,2022-07-17,Under the Amalfi Sun,2022-07-13,4.0,it,873127.0,"1 exact title, released before",1
9,Films (Non-English),Youth,NaN,2026-04-19,Youth,2026-03-19,31.0,ta,1542352.0,"1 exact title, released before",5



How often each rule decided:


rule
1 exact title, released before    29
no results                         1
Name: count, dtype: int64

In [70]:
# Flags worth checking by hand
non_english_list = matches["category"].str.contains("Non-English")
flags = pd.DataFrame(
    {
        "no match": matches["tmdb_id"].isna(),
        "low confidence (rule 3-4)": matches["rule"].str[0].isin(["3", "4"]),
        "released after charting": matches["days_before"].fillna(0) < 0,
        "film released 3+ years before charting": (~matches["category"].str.startswith("TV"))
        & (matches["days_before"].fillna(0) > 3 * 365),
        "English list, non-English title": ~non_english_list & matches["language"].ne("en") & matches["language"].notna(),
        "Non-English list, English title": non_english_list & matches["language"].eq("en"),
    }
)
print("Flag counts:")
display(flags.sum().to_frame("titles"))
flagged = matches[flags.any(axis=1)]
display(flagged[["category", "netflix_title", "tmdb_name", "tmdb_date", "days_before", "language", "rule"]])

Flag counts:


,titles
no match,1
low confidence (rule 3-4),0
released after charting,0
film released 3+ years before charting,2
"English list, non-English title",0
"Non-English list, English title",0


,category,netflix_title,tmdb_name,tmdb_date,days_before,language,rule
1,Films (English),Monsters vs. Aliens,Monsters vs Aliens,2009-03-19,6338.0,en,"1 exact title, released before"
5,Films (Non-English),Tadeo Jones 2: El secreto del Rey Midas,"Tad, the Lost Explorer, and the Secret of King Midas",2017-03-25,1926.0,es,"1 exact title, released before"
21,TV (English),Jamie Foxx: What Had Happened Was...,NaN,None,NaN,NaN,no results


## 6 · Do TMDB season runtimes agree with Netflix's?

Netflix's `runtime` is the total length of a film or a whole TV season, in hours. For matched TV titles, fetch the season from TMDB, add up its episode runtimes, and compare with Netflix's number (when Netflix published one, i.e. weeks after June 2023).

A ratio close to 1.0 is strong evidence that both the show **and** the season number are right. A ratio far from 1.0 points to a wrong match or a season-numbering mismatch.

In [71]:
checks = []
tv_sample = sample[sample["is_tv"]].merge(matches[["netflix_title", "season_title", "tmdb_id"]],
                                          left_on=["show_title", "season_title"],
                                          right_on=["netflix_title", "season_title"], how="left")
for _, row in tv_sample.iterrows():
    season_number, pattern = parse_season(row["show_title"], row["season_title"])
    record = {"netflix_title": row["show_title"], "season_title": row["season_title"],
              "season_number": season_number, "netflix_runtime_h": row["netflix_runtime"]}
    if pd.notna(row["tmdb_id"]) and season_number is not None:
        season = tmdb_get(f"/tv/{int(row['tmdb_id'])}/season/{season_number}")
        if season:
            runtimes = [ep.get("runtime") or 0 for ep in season.get("episodes", [])]
            record.update(
                episodes=len(runtimes),
                episodes_missing_runtime=sum(r == 0 for r in runtimes),
                tmdb_runtime_h=round(sum(runtimes) / 60, 2),
                season_air_date=season.get("air_date"),
            )
        else:
            record["note"] = "season not found on TMDB"
    else:
        record["note"] = "no match or no season number"
    checks.append(record)

checks = pd.DataFrame(checks)
if "tmdb_runtime_h" in checks:
    checks["ratio_tmdb_to_netflix"] = (checks["tmdb_runtime_h"] / checks["netflix_runtime_h"]).round(2)
display(checks)

,netflix_title,season_title,season_number,netflix_runtime_h,episodes,episodes_missing_runtime,tmdb_runtime_h,season_air_date,note,ratio_tmdb_to_netflix
0,Bridgerton,Bridgerton: Season 4,4.0,8.8833,8.0,0.0,8.90,2026-01-29,NaN,1.00
1,Perfect Match,Perfect Match: Season 2,2.0,9.6167,10.0,0.0,9.55,2024-06-07,NaN,0.99
2,Haunted Hotel,Haunted Hotel: Season 2,2.0,3.4167,8.0,0.0,3.35,2026-10-09,NaN,0.98
3,Kylie,Kylie: Limited Series,1.0,3.0667,3.0,0.0,3.10,2026-05-20,NaN,1.01
4,Unsolved Mysteries,Unsolved Mysteries: Volume 5,5.0,2.9667,55.0,55.0,0.00,1992-09-16,NaN,0.00
5,Daughter From Another Mother,Daughter From Another Mother: Season 2,2.0,NaN,8.0,0.0,4.90,2021-12-24,NaN,NaN
6,Tokyo Swindlers,Tokyo Swindlers: Season 1,1.0,6.1500,7.0,0.0,6.10,2024-07-25,NaN,0.99
7,Ganglands,Ganglands: Season 1,1.0,NaN,6.0,0.0,4.63,2021-09-24,NaN,NaN
8,Let's Get Divorced,Let's Get Divorced: Season 1,1.0,9.3667,9.0,0.0,9.33,2023-06-22,NaN,1.00
9,Just One Look,Just One Look: Limited Series,1.0,4.4833,6.0,0.0,4.45,2025-03-04,NaN,0.99


In [72]:
def season_from_tmdb(tmdb_id, label, first_week):
    # Find the season number using TMDB's own season list: by name first, then by air date.
    show = tmdb_get(f"/tv/{int(tmdb_id)}") or {}
    seasons = [s for s in show.get("seasons", []) if s.get("season_number", 0) > 0]  # skip "Specials"
    for s in seasons:
        name = (s.get("name") or "").casefold()
        if label and name and (label.casefold() in name or name in label.casefold()):
            return s["season_number"], f"name match: {s.get('name')}"
    started = [(pd.to_datetime(s.get("air_date"), errors="coerce"), s["season_number"], s.get("name"))
               for s in seasons]
    before = [x for x in started if pd.notna(x[0]) and x[0] <= first_week]
    if before:
        date, number, name = max(before)
        return number, f"latest season before charting: {name} ({date.date()})"
    return None, "not found"


others = tv[tv["season_pattern"] == "other"].sample(min(10, (tv["season_pattern"] == "other").sum()), random_state=1)
results = []
for _, row in others.iterrows():
    best, rule = pick_match(row, search_candidates(row))
    label = str(row["season_title"])[len(row["show_title"]):].strip(" :-")
    number, how = (None, "no show match") if best is None else season_from_tmdb(best["tmdb_id"], label, row["first_week"])
    results.append({"season_title": row["season_title"], "tmdb_show": None if best is None else best["name"],
                     "season_number": number, "how": how})
display(pd.DataFrame(results))

,season_title,tmdb_show,season_number,how
0,Funny AF with Kevin Hart: The Live Semifinals,Funny AF with Kevin Hart,1.0,latest season before charting: Season 1 (2026-04-20)
1,DAHMER: Monster: The Jeffrey Dahmer Story,DAHMER - Monster: The Jeffrey Dahmer Story,1.0,latest season before charting: Miniseries (2022-09-21)
2,How to Ruin Love: The Proposal,How to Ruin Love,1.0,name match: The Proposal
3,The Great British Baking Show: Collection 13,The Great British Bake Off,9.0,latest season before charting: Series 16 (2025-09-02)
4,The Great British Baking Show: Collection 14,The Great British Bake Off,10.0,latest season before charting: Series 17 (2026-09-22)
5,The Snow Girl: 2. The Soul Game,The Snow Girl,2.0,name match: The Soul Game
6,Royal Rumble 2025,NaN,NaN,no show match
7,STEEL BALL RUN JoJo’s Bizarre Adventure: 1st STAGE,JoJo's Bizarre Adventure,6.0,latest season before charting: STEEL BALL RUN (2026-03-19)
8,Magic and Muscles: Mashle: Magic And Muscles The Divine ...,MASHLE: MAGIC AND MUSCLES,1.0,latest season before charting: Season 1 (2023-04-08)
9,Larva in Mars: Larva In Mars,Larva in Mars,1.0,latest season before charting: Season 1 (2024-01-13)


## 7 · Findings and decisions

 - show-level match for all titles; season details only when the season is clear or its name matches; otherwise leave blank.
 - all but one were within ±15% of runtime measurement


